# 04 — Async and Concurrency

`async def` vs `def` route handlers, what FastAPI actually does with each, the classic "blocking call in an async function" footgun, and concurrent outbound requests with `httpx.AsyncClient` — the async fundamentals interviewers check once they know you understand the basic routing/validation API.

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*httpx2.*")   # silence a harmless TestClient/httpx notice

import asyncio
import time
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

## 1. `async def` vs `def` — what FastAPI does with each

- **`async def` route** — run directly on the event loop. Correct when everything inside is `await`-compatible (async DB drivers, `httpx.AsyncClient`, `asyncio.sleep`) — lets one worker serve many concurrent requests while each is waiting on I/O.
- **`def` (sync) route** — FastAPI automatically runs it in a **separate thread pool**, not on the event loop, specifically so a blocking synchronous call (e.g. `requests.get`, a sync DB driver) doesn't stall the whole event loop for every other request.

**The interview trap:** writing `async def` and then calling a **blocking** function inside it (e.g. `time.sleep()`, `requests.get()`, a sync DB call) — that blocks the *entire* event loop, freezing every other concurrent request being served by that worker, which a plain `def` route would not have done.

In [2]:
@app.get("/bad-async")
async def bad_async_route():
    time.sleep(0.05)          # BLOCKING call inside async def -- freezes the event loop
    return {"done": True}

@app.get("/good-async")
async def good_async_route():
    await asyncio.sleep(0.05)  # non-blocking -- yields control back to the event loop
    return {"done": True}

@app.get("/sync-route")
def sync_route():
    time.sleep(0.05)          # fine here -- FastAPI already runs sync defs in a thread pool
    return {"done": True}

client = TestClient(app)
print(client.get("/bad-async").json())
print(client.get("/good-async").json())
print(client.get("/sync-route").json())
# all three return correct results in this single-request demo --
# the difference only shows up under CONCURRENT load, illustrated next

{'done': True}
{'done': True}
{'done': True}


## 2. Demonstrating the blocking-call problem under concurrency

Below, 5 concurrent requests hit a route that sleeps for 100ms each. With a truly non-blocking `await asyncio.sleep`, 5 concurrent requests should complete in roughly the time of **one** (they overlap); with a blocking `time.sleep` inside `async def`, they serialize and take roughly **5x** as long, because the blocked event loop can't even start the next one until the first finishes.

In [3]:
import httpx
import uvicorn
import threading

server_app = FastAPI()

@server_app.get("/blocking")
async def blocking_endpoint():
    time.sleep(0.1)
    return {"done": True}

@server_app.get("/non-blocking")
async def non_blocking_endpoint():
    await asyncio.sleep(0.1)
    return {"done": True}

config = uvicorn.Config(server_app, host="127.0.0.1", port=8123, log_level="warning")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(0.5)   # give the server a moment to start listening

In [4]:
async def time_concurrent_requests(path, n=5):
    async with httpx.AsyncClient(base_url="http://127.0.0.1:8123") as ac:
        start = time.perf_counter()
        await asyncio.gather(*[ac.get(path) for _ in range(n)])
        return time.perf_counter() - start

blocking_time = await time_concurrent_requests("/blocking")
non_blocking_time = await time_concurrent_requests("/non-blocking")

print(f"5 concurrent requests to a BLOCKING async route:     {blocking_time*1000:.0f} ms")
print(f"5 concurrent requests to a NON-blocking async route: {non_blocking_time*1000:.0f} ms")
print(f"blocking is ~{blocking_time/non_blocking_time:.1f}x slower under concurrency")

5 concurrent requests to a BLOCKING async route:     513 ms
5 concurrent requests to a NON-blocking async route: 113 ms
blocking is ~4.6x slower under concurrency


In [5]:
server.should_exit = True
thread.join(timeout=2)

## 3. Concurrent outbound calls with `httpx.AsyncClient`

A route that needs to call **multiple** downstream services should issue those calls concurrently with `asyncio.gather`, not sequentially with repeated `await` — the same total wait time as the *slowest* call, instead of the *sum* of all of them.

In [6]:
async def fetch_user_profile():
    await asyncio.sleep(0.05)
    return {"name": "alice"}

async def fetch_user_orders():
    await asyncio.sleep(0.05)
    return [{"order_id": 1}, {"order_id": 2}]

async def sequential():
    start = time.perf_counter()
    profile = await fetch_user_profile()
    orders = await fetch_user_orders()
    return time.perf_counter() - start

async def concurrent():
    start = time.perf_counter()
    profile, orders = await asyncio.gather(fetch_user_profile(), fetch_user_orders())
    return time.perf_counter() - start

seq_time = await sequential()
conc_time = await concurrent()
print(f"sequential: {seq_time*1000:.0f} ms, concurrent: {conc_time*1000:.0f} ms")

sequential: 101 ms, concurrent: 50 ms


## 4. Interview Q&A

1. **"Should every FastAPI route be `async def`?"** — no; only if everything inside is genuinely non-blocking. A sync route calling a blocking library is *safer* as a plain `def` (FastAPI runs it in a thread pool) than as `async def` with a blocking call hidden inside it.
2. **"What actually breaks if you call `time.sleep()` inside an `async def` route?"** — it blocks the single event loop thread, so *every other concurrent request* on that worker stalls until the sleep finishes — not just the one request that called it.
3. **"How do you call three downstream APIs from one route as fast as possible?"** — `await asyncio.gather(call1(), call2(), call3())` so they run concurrently; awaiting them one after another sequentially sums their latencies instead of overlapping.
4. **"Why does FastAPI run sync `def` routes in a thread pool instead of directly on the event loop?"** — so a blocking call inside a sync handler only ties up one thread pool worker, not the single event loop that every concurrent request depends on.

## Summary

- `async def` only pays off when everything inside actually `await`s; a blocking call inside one freezes every concurrent request on that worker — verified above with real timing.
- Plain `def` routes are automatically offloaded to a thread pool by FastAPI, which is why a blocking library call is *safer* there than inside a careless `async def`.
- `asyncio.gather` runs independent awaitables concurrently — use it whenever a route fans out to multiple downstream calls.
- Next: `05_data_engineering_patterns_with_fastapi.ipynb`.